# 工具系统集成指南

**前置知识**: Function Calling、Tool Registry、Tool Executor、Structured Output

**学习目标**: 将所有组件整合，构建完整的 AI Agent 工具调用系统

---

## 系统架构

```
用户输入 → LLM → 结构化输出解析 → 函数调用解析 → 验证 → 执行器 → 结果
                      ↑                              ↓
                 OutputSchema              ToolRegistry
```

**四大组件**:
1. **Function Calling**: 定义函数 Schema，解析 LLM 输出
2. **Tool Registry**: 注册和管理工具
3. **Tool Executor**: 安全执行工具
4. **Structured Output**: 解析和验证结构化输出

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
import json
from typing import List, Optional
from pydantic import BaseModel, Field

sys.path.insert(0, '..')

from src.function_calling import (
    FunctionDefinition,
    FunctionParameter,
    FunctionCall,
    FunctionCallParser,
    ParameterType,
)
from src.tool_registry import ToolRegistry, Tool
from src.tool_executor import ToolExecutor, ExecutionContext, ExecutionStatus
from src.structured_output import (
    StructuredOutputParser,
    OutputSchema,
    ValidationError,
    create_choice_parser,
)

print("所有组件导入成功！")

---

## 第一步：构建工具集

创建一个实用的工具集，模拟真实 Agent 场景。

In [ ]:
# ============================================================
# 创建工具注册表
# ============================================================
registry = ToolRegistry()

# 数学工具
@registry.register(tags=["math", "utility"])
def calculator(expression: str) -> float:
    """计算数学表达式，支持 +, -, *, /, ** 等运算"""
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        raise ValueError("表达式包含非法字符")
    return eval(expression)

# 天气工具
@registry.register(tags=["weather", "api"])
def get_weather(city: str, unit: str = "celsius") -> dict:
    """获取城市天气信息"""
    data = {
        "北京": {"temp": 25, "condition": "晴", "humidity": 40},
        "上海": {"temp": 28, "condition": "多云", "humidity": 65},
        "广州": {"temp": 32, "condition": "雨", "humidity": 80},
    }
    weather = data.get(city, {"temp": 20, "condition": "未知", "humidity": 50})
    if unit == "fahrenheit":
        weather["temp"] = weather["temp"] * 9/5 + 32
    weather["city"] = city
    weather["unit"] = unit
    return weather

# 搜索工具
@registry.register(tags=["search", "api"])
def web_search(query: str, max_results: int = 5) -> list:
    """搜索互联网信息"""
    return [
        {"title": f"结果 {i}: {query}", "url": f"https://example.com/{i}"}
        for i in range(1, min(max_results + 1, 11))
    ]

# 文本工具
@registry.register(tags=["text", "utility"])
def text_analysis(text: str) -> dict:
    """分析文本统计信息"""
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()) or 1,
    }

print(f"已注册 {len(registry)} 个工具:")
for tool in registry.list_tools():
    print(f"  - {tool.name}: {tool.description}")

---

## 第二步：创建执行器和解析器

In [ ]:
# ============================================================
# 创建执行器
# ============================================================
executor = ToolExecutor(
    registry=registry,
    default_timeout=10.0,
    max_retries=2,
)

# ============================================================
# 创建函数调用解析器
# ============================================================
function_defs = [tool.to_function_definition() for tool in registry.list_tools()]
call_parser = FunctionCallParser(function_defs)

print(f"执行器: {executor}")
print(f"解析器已注册 {len(function_defs)} 个函数")

---

## 第三步：定义 Agent 输出结构

使用 Pydantic 定义 LLM 应该输出的结构。

In [ ]:
# ============================================================
# 定义 Agent 输出结构
# ============================================================

class AgentThought(BaseModel):
    """Agent 的思考过程"""
    reasoning: str = Field(description="推理过程")
    action: str = Field(description="决定采取的行动: 'tool_call' 或 'final_answer'")

class ToolCallRequest(BaseModel):
    """工具调用请求"""
    thought: AgentThought
    tool_name: str = Field(description="要调用的工具名称")
    arguments: dict = Field(description="工具参数")

class FinalAnswer(BaseModel):
    """最终回答"""
    thought: AgentThought
    answer: str = Field(description="给用户的最终回答")

# 创建解析器
tool_call_parser = StructuredOutputParser(ToolCallRequest)
final_answer_parser = StructuredOutputParser(FinalAnswer)

print("Agent 输出结构已定义")

---

## 第四步：构建 Agent 类

In [ ]:
# ============================================================
# Agent 类
# ============================================================

class SimpleAgent:
    """简单的工具调用 Agent"""
    
    def __init__(self, registry: ToolRegistry, executor: ToolExecutor):
        self.registry = registry
        self.executor = executor
        self.call_parser = FunctionCallParser(
            [t.to_function_definition() for t in registry.list_tools()]
        )
        self.history = []  # 对话历史
    
    def get_system_prompt(self) -> str:
        """生成系统提示词"""
        tools_desc = self.registry.get_tool_descriptions()
        return f"""你是一个智能助手，可以使用以下工具:

{tools_desc}

当需要使用工具时，输出 JSON 格式:
```json
{{"name": "工具名", "arguments": {{...}}}}
```
"""
    
    def process_llm_output(self, output: str) -> dict:
        """处理 LLM 输出"""
        # 尝试解析工具调用
        calls = self.call_parser.parse(output)
        
        if not calls:
            return {"type": "text", "content": output}
        
        call = calls[0]
        
        # 验证
        errors = self.call_parser.validate(call)
        if errors:
            return {"type": "error", "content": f"验证失败: {errors}"}
        
        # 执行
        result = self.executor.execute_call(call)
        
        return {
            "type": "tool_result",
            "tool_name": call.name,
            "arguments": call.arguments,
            "success": result.is_success,
            "output": result.output if result.is_success else result.error,
        }
    
    def run(self, user_input: str, llm_response: str) -> str:
        """运行一轮对话"""
        self.history.append({"role": "user", "content": user_input})
        
        result = self.process_llm_output(llm_response)
        
        if result["type"] == "tool_result":
            response = f"工具 {result['tool_name']} 执行{'成功' if result['success'] else '失败'}\n"
            response += f"结果: {result['output']}"
        else:
            response = result["content"]
        
        self.history.append({"role": "assistant", "content": response})
        return response

# 创建 Agent
agent = SimpleAgent(registry, executor)
print("Agent 已创建")
print("\n系统提示词:")
print(agent.get_system_prompt())

---

## 第五步：测试 Agent

In [ ]:
# ============================================================
# 测试 Agent
# ============================================================

test_cases = [
    {
        "user": "北京今天天气怎么样？",
        "llm": '{"name": "get_weather", "arguments": {"city": "北京"}}',
    },
    {
        "user": "计算 2 的 10 次方",
        "llm": '{"name": "calculator", "arguments": {"expression": "2**10"}}',
    },
    {
        "user": "搜索 Python 教程",
        "llm": '{"name": "web_search", "arguments": {"query": "Python教程", "max_results": 3}}',
    },
    {
        "user": "分析这段文字: Hello World",
        "llm": '{"name": "text_analysis", "arguments": {"text": "Hello World"}}',
    },
]

print("=== Agent 测试 ===")
for i, case in enumerate(test_cases, 1):
    print(f"\n--- 测试 {i} ---")
    print(f"用户: {case['user']}")
    response = agent.run(case['user'], case['llm'])
    print(f"Agent: {response}")

---

## 第六步：高级功能 - 多工具调用

In [ ]:
# ============================================================
# 多工具调用
# ============================================================

def process_multi_tool_calls(llm_output: str) -> list:
    """处理多个工具调用"""
    calls = call_parser.parse(llm_output)
    
    if not calls:
        return [{"type": "text", "content": llm_output}]
    
    # 批量执行
    results = executor.execute_batch(calls, parallel=True)
    
    return [
        {
            "tool": call.name,
            "args": call.arguments,
            "success": result.is_success,
            "output": result.output if result.is_success else result.error,
        }
        for call, result in zip(calls, results)
    ]

# 模拟多工具调用
multi_call_output = '''[
    {"name": "get_weather", "arguments": {"city": "北京"}},
    {"name": "get_weather", "arguments": {"city": "上海"}},
    {"name": "calculator", "arguments": {"expression": "100+200"}}
]'''

# 解析多个调用
import re
json_objects = re.findall(r'\{[^{}]+\}', multi_call_output)
calls = []
for obj in json_objects:
    parsed = call_parser.parse(obj)
    if parsed:
        calls.extend(parsed)

print(f"解析到 {len(calls)} 个调用")

# 批量执行
results = executor.execute_batch(calls, parallel=True)
for call, result in zip(calls, results):
    print(f"\n{call.name}({call.arguments}):")
    print(f"  结果: {result.output if result.is_success else result.error}")

---

## 第七步：结构化输出与工具调用结合

In [ ]:
# ============================================================
# 结构化输出解析
# ============================================================

class AgentResponse(BaseModel):
    """Agent 响应结构"""
    thinking: str = Field(description="思考过程")
    action_type: str = Field(description="行动类型: tool_call 或 respond")
    tool_call: Optional[dict] = Field(default=None, description="工具调用信息")
    response: Optional[str] = Field(default=None, description="直接回复")

response_parser = StructuredOutputParser(AgentResponse)

# 模拟 LLM 输出
llm_output = '''```json
{
    "thinking": "用户想知道北京天气，我需要调用天气工具",
    "action_type": "tool_call",
    "tool_call": {"name": "get_weather", "arguments": {"city": "北京"}},
    "response": null
}
```'''

# 解析
parsed = response_parser.parse(llm_output)
print(f"思考: {parsed.thinking}")
print(f"行动: {parsed.action_type}")

if parsed.action_type == "tool_call" and parsed.tool_call:
    call = FunctionCall(
        name=parsed.tool_call["name"],
        arguments=parsed.tool_call["arguments"],
    )
    result = executor.execute_call(call)
    print(f"工具结果: {result.output}")

---

## 第八步：完整的 ReAct 风格 Agent

In [ ]:
# ============================================================
# ReAct 风格 Agent
# ============================================================

class ReActAgent:
    """ReAct (Reasoning + Acting) 风格的 Agent"""
    
    def __init__(self, registry: ToolRegistry, executor: ToolExecutor):
        self.registry = registry
        self.executor = executor
        self.parser = FunctionCallParser(
            [t.to_function_definition() for t in registry.list_tools()]
        )
        self.max_iterations = 5
    
    def step(self, thought: str, action: str, action_input: dict) -> str:
        """执行一步 ReAct 循环"""
        print(f"Thought: {thought}")
        print(f"Action: {action}")
        print(f"Action Input: {action_input}")
        
        if action == "finish":
            return action_input.get("answer", "完成")
        
        # 执行工具
        result = self.executor.execute(action, action_input)
        observation = result.output if result.is_success else f"Error: {result.error}"
        print(f"Observation: {observation}")
        return observation
    
    def run(self, steps: list) -> str:
        """运行多步 ReAct 循环"""
        print("=== ReAct Agent 运行 ===")
        
        for i, step_data in enumerate(steps, 1):
            print(f"\n--- Step {i} ---")
            result = self.step(
                step_data["thought"],
                step_data["action"],
                step_data["action_input"],
            )
            if step_data["action"] == "finish":
                print(f"\nFinal Answer: {result}")
                return result
        
        return "达到最大迭代次数"

# 创建 ReAct Agent
react_agent = ReActAgent(registry, executor)

# 模拟 ReAct 循环
steps = [
    {
        "thought": "用户想比较北京和上海的天气，我需要先获取北京天气",
        "action": "get_weather",
        "action_input": {"city": "北京"},
    },
    {
        "thought": "已获取北京天气(25°C)，现在获取上海天气",
        "action": "get_weather",
        "action_input": {"city": "上海"},
    },
    {
        "thought": "已获取两地天气，北京25°C，上海28°C，可以回答用户了",
        "action": "finish",
        "action_input": {"answer": "北京25°C(晴)，上海28°C(多云)，上海比北京热3度"},
    },
]

react_agent.run(steps)

---

## 第九步：执行统计与监控

In [ ]:
# ============================================================
# 执行统计
# ============================================================

stats = executor.get_stats()
print("=== 执行统计 ===")
print(f"总执行次数: {stats['total_executions']}")
print(f"成功率: {stats['success_rate']:.1%}")
print(f"平均耗时: {stats['average_time']*1000:.2f} ms")

# 按工具统计
print("\n=== 按工具统计 ===")
for tool in registry.list_tools():
    tool_history = executor.get_history(tool_name=tool.name)
    if tool_history:
        success = sum(1 for r in tool_history if r.is_success)
        print(f"{tool.name}: {len(tool_history)} 次, 成功率 {success/len(tool_history):.0%}")

---

## 本节要点

**集成架构**:
```
ToolRegistry → FunctionCallParser → ToolExecutor → ExecutionResult
     ↓              ↓                    ↓              ↓
  工具管理      解析LLM输出          安全执行        结果追踪
```

**关键集成点**:
1. `registry.list_tools()` → `FunctionCallParser` 同步函数定义
2. `parser.parse()` → `executor.execute_call()` 解析后执行
3. `StructuredOutputParser` + `FunctionCall` 结合使用

**最佳实践**:
- 使用 `execute_batch(parallel=True)` 并行执行多个调用
- 通过 `get_stats()` 监控执行情况
- 使用生命周期钩子添加日志和监控